# 05 — Model Training

**Thesis:** Comparative Analysis of Machine Learning Algorithms for Predicting CVD Risk in a Tunisian Hospital Population

## Objective
This notebook trains and tunes the four algorithms required by the comparative analysis:
1. Decision Tree
2. Random Forest
3. Multinomial Logistic Regression
4. XGBoost

Each model is wrapped in an `imblearn.pipeline.Pipeline` that nests preprocessing and SMOTENC inside stratified cross-validation, preventing any resampling-related data leakage. Hyperparameters are tuned with `GridSearchCV` scoring on `f1_macro`. **The test set created in `04_ML_Preprocessing.ipynb` is never accessed in this notebook.**

## Why SMOTENC, not SMOTE
This dataset contains five binary/ordinal variables encoded as integers. Applying plain SMOTE would generate fractional synthetic values for these columns (e.g., Sex = 0.37), which are not valid categories. SMOTENC avoids this by using majority-vote among K nearest neighbours for categorical columns and standard continuous interpolation for numerical columns.

## Data-leakage protection
The `imblearn.pipeline.Pipeline` (not sklearn's) ensures that:
- SMOTENC's `fit_resample` is called only on the training portion of each CV fold.
- SMOTENC is **not** called during `predict()` (i.e., on validation or test data).
- Scaling and imputation are also fit on the training fold only.

## Required packages
`scikit-learn == 1.4.2`, `imbalanced-learn == 0.12.3`, `xgboost == 3.4.1`

## 0. Setup

In [1]:
import json
import warnings
from math import prod
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_DIR = Path.cwd().parent
ARTIFACTS_DIR = PROJECT_DIR / "ml_artifacts"
SPLITS_DIR = ARTIFACTS_DIR / "splits"
MODELS_DIR = ARTIFACTS_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Artifacts dir:", ARTIFACTS_DIR)
print("Splits dir   :", SPLITS_DIR)
print("Models dir   :", MODELS_DIR)


Artifacts dir: /home/claude/ML_HOSPITAL/ml_artifacts
Splits dir   : /home/claude/ML_HOSPITAL/ml_artifacts/splits
Models dir   : /home/claude/ML_HOSPITAL/ml_artifacts/models


In [2]:
# --- imblearn ---
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTENC
import imblearn
print("imbalanced-learn:", imblearn.__version__)

# --- xgboost ---
from xgboost import XGBClassifier
import xgboost
print("xgboost:", xgboost.__version__)

# --- sklearn ---
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
print("scikit-learn:", sklearn.__version__)


imbalanced-learn: 0.12.3
xgboost: 3.4.1
scikit-learn: 1.4.2


## 1. Load Training Artifacts

In [3]:
X_train = pd.read_csv(SPLITS_DIR / "X_train.csv", index_col=0)
y_train = pd.read_csv(SPLITS_DIR / "y_train.csv", index_col=0).iloc[:, 0]

with open(ARTIFACTS_DIR / "ml_config.json") as f:
    config = json.load(f)

TARGET = config["target"]
RISK_LABELS = {int(k): v for k, v in config["risk_labels"].items()}
CONTINUOUS_NUMERIC = config["continuous_numeric"]
BINARY_ORDINAL = config["binary_ordinal"]
FEATURES = config["features"]

assert X_train.shape == (1223, 14), f"Unexpected X_train shape: {X_train.shape}"
assert y_train.shape == (1223,),    f"Unexpected y_train shape: {y_train.shape}"

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("Training class distribution:")
print(y_train.value_counts().reindex([0, 1, 2]).rename(index=RISK_LABELS))


X_train: (1223, 14)  y_train: (1223,)
Training class distribution:
CVD Risk Level
LOW             176
INTERMEDIARY    465
HIGH            582
Name: count, dtype: int64


## 2. Cross-Validation Strategy and Scoring

5-fold Stratified K-Fold with `shuffle=True` and a fixed `random_state` guarantees reproducible folds while preserving class proportions in every fold. Primary tuning metric: `f1_macro`.

In [4]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
SCORING = "f1_macro"

print("CV strategy:", cv_strategy)
print("Tuning metric:", SCORING)


CV strategy: StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
Tuning metric: f1_macro


## 3. Pipeline Construction

Two pipeline builders are defined:
- **`build_tree_pipeline(estimator)`** -- for tree-based models (Decision Tree, Random Forest, XGBoost): `SimpleImputer -> SMOTENC -> classifier`. No scaling; trees are invariant to monotonic transformations.
- **`build_lr_pipeline(estimator)`** -- for Logistic Regression: `SimpleImputer -> ColumnTransformer(scale continuous) -> SMOTENC -> classifier`. The `ColumnTransformer` scales only the 9 continuous variables; the 5 binary/ordinal variables pass through unchanged.

**SMOTENC configuration:** `categorical_features=BINARY_ORDINAL` (column names, works because `SimpleImputer.set_output(transform="pandas")` passes a DataFrame downstream), `k_neighbors=5`.

`imblearn.pipeline.Pipeline` is used (not `sklearn.pipeline.Pipeline`) because only the imblearn version correctly calls `fit_resample` during fitting and skips resampling during prediction.

In [5]:
assert all(c in X_train.columns for c in BINARY_ORDINAL), \
    "BINARY_ORDINAL column mismatch - check ml_config.json."
assert all(c in X_train.columns for c in CONTINUOUS_NUMERIC), \
    "CONTINUOUS_NUMERIC column mismatch - check ml_config.json."
print("Feature column check: PASSED")


Feature column check: PASSED


In [6]:
def _smotenc():
    # Returns a fresh SMOTENC instance with the fixed categorical feature list.
    return SMOTENC(
        categorical_features=BINARY_ORDINAL,
        random_state=RANDOM_SEED,
        k_neighbors=5,
    )

def build_tree_pipeline(estimator):
    # Imputation -> SMOTENC -> tree classifier (no scaling).
    return Pipeline(steps=[
        ("impute", SimpleImputer(strategy="median").set_output(transform="pandas")),
        ("smote", _smotenc()),
        ("clf", estimator),
    ])

def build_lr_pipeline(estimator):
    # Imputation -> scale continuous -> SMOTENC -> logistic classifier.
    ct = ColumnTransformer(
        transformers=[("scale", StandardScaler(), CONTINUOUS_NUMERIC)],
        remainder="passthrough",
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")
    return Pipeline(steps=[
        ("impute", SimpleImputer(strategy="median").set_output(transform="pandas")),
        ("scale", ct),
        ("smote", _smotenc()),
        ("clf", estimator),
    ])

def run_grid_search(name, pipeline, param_grid, n_jobs_grid=-1):
    # Runs GridSearchCV, prints best score/params, returns the fitted GridSearchCV.
    n_combinations = prod(len(v) for v in param_grid.values())
    grid = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        scoring=SCORING,
        cv=cv_strategy,
        n_jobs=n_jobs_grid,
        refit=True,
        error_score="raise",
        return_train_score=False,
    )
    print(f"[{name}] fitting GridSearchCV ({n_combinations} param combinations x 5 folds) ...")
    grid.fit(X_train, y_train)
    cv_std = grid.cv_results_["std_test_score"][grid.best_index_]
    print(f"[{name}] best CV Macro F1 : {grid.best_score_:.4f}  (+/-{cv_std:.4f})")
    print(f"[{name}] best params      : {grid.best_params_}")
    return grid

print("Pipeline builders and run_grid_search defined.")


Pipeline builders and run_grid_search defined.


## 4. Model 1 — Decision Tree

A single tree recursively splits the feature space on the variable/threshold that minimises impurity. It requires no scaling, handles mixed variable types naturally, and produces fully interpretable rules.

**Tuned hyperparameters:** `max_depth`, `min_samples_leaf`, `criterion`.

In [7]:
dt_param_grid = {
    "clf__max_depth":        [3, 5, 7, 10, None],
    "clf__min_samples_leaf": [1, 5, 10, 20],
    "clf__criterion":        ["gini", "entropy"],
}

dt_pipeline = build_tree_pipeline(
    DecisionTreeClassifier(random_state=RANDOM_SEED)
)

dt_grid = run_grid_search("Decision Tree", dt_pipeline, dt_param_grid, n_jobs_grid=-1)


[Decision Tree] fitting GridSearchCV (40 param combinations x 5 folds) ...


[Decision Tree] best CV Macro F1 : 0.5358  (+/-0.0296)
[Decision Tree] best params      : {'clf__criterion': 'gini', 'clf__max_depth': 7, 'clf__min_samples_leaf': 5}


## 5. Model 2 — Random Forest

An ensemble of trees, each trained on a bootstrap sample and random feature subsets.

**Note on parallelism:** `RandomForestClassifier(n_jobs=1)` is used inside the pipeline because `GridSearchCV` already uses `n_jobs=-1` for outer parallelism.

**Tuned hyperparameters:** `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`.

In [8]:
rf_param_grid = {
    "clf__n_estimators":     [200, 400],
    "clf__max_depth":        [None, 10, 20],
    "clf__min_samples_leaf": [1, 5, 10],
    "clf__max_features":     ["sqrt", "log2"],
}

rf_pipeline = build_tree_pipeline(
    RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=1)
)

rf_grid = run_grid_search("Random Forest", rf_pipeline, rf_param_grid, n_jobs_grid=-1)


[Random Forest] fitting GridSearchCV (36 param combinations x 5 folds) ...


[Random Forest] best CV Macro F1 : 0.6005  (+/-0.0188)
[Random Forest] best params      : {'clf__max_depth': 10, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 10, 'clf__n_estimators': 200}


## 6. Model 3 — Multinomial Logistic Regression

Models class probabilities via softmax over a linear combination of (standardized) features. Provides an interpretable linear baseline.

**Implementation note:** with `solver="lbfgs"` and a multi-class target, scikit-learn 1.4.x automatically fits a multinomial (softmax) model.

**Tuned hyperparameter:** `C`.

In [9]:
lr_param_grid = {
    "clf__C": [0.01, 0.1, 1.0, 10.0, 100.0],
}

lr_pipeline = build_lr_pipeline(
    LogisticRegression(
        random_state=RANDOM_SEED,
        max_iter=5000,
        solver="lbfgs",
    )
)

lr_grid = run_grid_search("Logistic Regression", lr_pipeline, lr_param_grid, n_jobs_grid=-1)


[Logistic Regression] fitting GridSearchCV (5 param combinations x 5 folds) ...


[Logistic Regression] best CV Macro F1 : 0.5087  (+/-0.0389)
[Logistic Regression] best params      : {'clf__C': 100.0}


## 7. Model 4 — XGBoost

A gradient-boosted ensemble built sequentially: each tree corrects the residual errors of the current ensemble, with regularization via depth limits, learning-rate shrinkage, and subsampling.

**Note on parallelism:** `XGBClassifier(n_jobs=1)` to avoid oversubscription with `GridSearchCV(n_jobs=-1)`.

**Tuned hyperparameters:** `n_estimators`, `max_depth`, `learning_rate`, `subsample`.

In [10]:
xgb_param_grid = {
    "clf__n_estimators":  [200, 400],
    "clf__max_depth":     [3, 5, 7],
    "clf__learning_rate": [0.01, 0.1, 0.2],
    "clf__subsample":     [0.8, 1.0],
}

xgb_pipeline = build_tree_pipeline(
    XGBClassifier(
        random_state=RANDOM_SEED,
        eval_metric="mlogloss",
        n_jobs=1,
    )
)

xgb_grid = run_grid_search("XGBoost", xgb_pipeline, xgb_param_grid, n_jobs_grid=-1)


[XGBoost] fitting GridSearchCV (36 param combinations x 5 folds) ...


[XGBoost] best CV Macro F1 : 0.5838  (+/-0.0277)
[XGBoost] best params      : {'clf__learning_rate': 0.1, 'clf__max_depth': 5, 'clf__n_estimators': 200, 'clf__subsample': 1.0}


## 8. Cross-Validation Results Summary

The table below reports each model's **cross-validated** Macro F1 on the training set only -- a provisional comparison for hyperparameter selection, not the final verdict. The authoritative comparison uses the completely untouched test set in `06_Model_Evaluation.ipynb`.

In [11]:
grids = {
    "Decision Tree":                    dt_grid,
    "Random Forest":                    rf_grid,
    "Multinomial Logistic Regression":  lr_grid,
    "XGBoost":                          xgb_grid,
}

cv_rows = []
for name, grid in grids.items():
    best_idx = grid.best_index_
    cv_rows.append({
        "Model": name,
        "Best CV Macro F1": round(grid.best_score_, 4),
        "CV Std":           round(grid.cv_results_["std_test_score"][best_idx], 4),
        "Best Params":      grid.best_params_,
    })

cv_summary = (
    pd.DataFrame(cv_rows)
    .sort_values("Best CV Macro F1", ascending=False)
    .reset_index(drop=True)
)
print("CV Tuning Summary (training-set cross-validation only -- NOT final evaluation):")
display(cv_summary[["Model", "Best CV Macro F1", "CV Std"]])

print("\nBest hyperparameters per model:")
for row in cv_rows:
    print(f"  {row['Model']:35s}: {row['Best Params']}")


CV Tuning Summary (training-set cross-validation only -- NOT final evaluation):


,Model,Best CV Macro F1,CV Std
0,Random Forest,0.6005,0.0188
1,XGBoost,0.5838,0.0277
2,Decision Tree,0.5358,0.0296
3,Multinomial Logistic Regression,0.5087,0.0389



Best hyperparameters per model:
  Decision Tree                      : {'clf__criterion': 'gini', 'clf__max_depth': 7, 'clf__min_samples_leaf': 5}
  Random Forest                      : {'clf__max_depth': 10, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 10, 'clf__n_estimators': 200}
  Multinomial Logistic Regression    : {'clf__C': 100.0}
  XGBoost                            : {'clf__learning_rate': 0.1, 'clf__max_depth': 5, 'clf__n_estimators': 200, 'clf__subsample': 1.0}


In [12]:
cv_summary.to_csv(ARTIFACTS_DIR / "cv_tuning_summary.csv", index=False)
print("CV summary saved ->", ARTIFACTS_DIR / "cv_tuning_summary.csv")


CV summary saved -> /home/claude/ML_HOSPITAL/ml_artifacts/cv_tuning_summary.csv


## 9. Save Fitted Pipelines

Each `GridSearchCV.best_estimator_` is a complete, fitted `imblearn` pipeline (imputation -> scaling if applicable -> SMOTENC -> classifier), refit on the **entire training set** with the best hyperparameters. These are the objects loaded and evaluated in `06_Model_Evaluation.ipynb`.

In [13]:
model_keys = {
    "Decision Tree":                   "decision_tree",
    "Random Forest":                   "random_forest",
    "Multinomial Logistic Regression": "logistic_regression",
    "XGBoost":                         "xgboost",
}

for display_name, file_key in model_keys.items():
    path = MODELS_DIR / f"{file_key}.joblib"
    joblib.dump(grids[display_name].best_estimator_, path)
    print(f"Saved -> {path}")


Saved -> /home/claude/ML_HOSPITAL/ml_artifacts/models/decision_tree.joblib
Saved -> /home/claude/ML_HOSPITAL/ml_artifacts/models/random_forest.joblib
Saved -> /home/claude/ML_HOSPITAL/ml_artifacts/models/logistic_regression.joblib
Saved -> /home/claude/ML_HOSPITAL/ml_artifacts/models/xgboost.joblib


## 10. Quick Sanity Check — Predict on Training Set

A model fitted to all the training data should achieve high training-set accuracy. This is expected (not alarming) and confirms the saved pipelines produce predictions without errors. Training performance is **not** a measure of generalization.

In [14]:
from sklearn.metrics import accuracy_score, f1_score

print("Training-set self-prediction (sanity check -- not generalization estimate):")
print(f"{'Model':35s} {'Train Acc':>10} {'Train Macro F1':>14}")
print("-" * 64)
for display_name, file_key in model_keys.items():
    model = joblib.load(MODELS_DIR / f"{file_key}.joblib")
    y_pred_train = model.predict(X_train)
    acc = accuracy_score(y_train, y_pred_train)
    f1 = f1_score(y_train, y_pred_train, average="macro")
    print(f"{display_name:35s} {acc:10.4f} {f1:14.4f}")

print("\nAll four pipelines load and predict correctly.")


Training-set self-prediction (sanity check -- not generalization estimate):
Model                                Train Acc Train Macro F1
----------------------------------------------------------------
Decision Tree                           0.7318         0.7108
Random Forest                           0.8217         0.7837
Multinomial Logistic Regression         0.5797         0.5270
XGBoost                                 0.9984         0.9981

All four pipelines load and predict correctly.


## Summary

| Step | Detail |
|---|---|
| Resampling | SMOTENC (column-name-based categorical specification; no invalid synthetic values) |
| Leakage protection | `imblearn.Pipeline` restricts SMOTENC to training folds only |
| CV strategy | `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` |
| Tuning metric | `f1_macro` |
| Models trained | Decision Tree, Random Forest, Logistic Regression, XGBoost |
| Test set accessed | No |

**What comes next:** `06_Model_Evaluation.ipynb` loads the four fitted pipelines and the untouched test set, and reports the final comparative evaluation.